In [25]:
import numpy as np
import pandas as pd
from pygments.lexers.textfmts import TodotxtLexer

#csv inventories
df_csv_inventories = pd.read_csv('inventories.csv')
#csv sales
df_csv_sales = pd.read_csv('sales.csv')
#csv satisfaction
df_csv_satisfaction = pd.read_csv('satisfaction.csv')

#implementamos los dropna
inventories = df_csv_inventories.dropna()
sales =  df_csv_sales.dropna()
satisfaction =  df_csv_satisfaction.dropna()

#################################################
### Sales

#Calculamos la venta total de las tiendas.
productos_vendidos =  sales.groupby("Producto")["Cantidad_Vendida"].sum()
productos_vendidos_tienda =  sales.groupby("ID_Tienda")["Cantidad_Vendida"].sum()
print("Cantidad de productos vendidos:\n",productos_vendidos.to_string())

print("Cantidad de productos vendidos , clasificado por tienda:\n",productos_vendidos_tienda.to_string())

#Calculamos los ingresos totales por tienda
sales["Venta_Total"] = sales["Cantidad_Vendida"] * sales["Precio_Unitario"]
ventas_tienda = sales.groupby("ID_Tienda")["Venta_Total"].sum()
print("Calculo ingresos por tienda:\n",ventas_tienda.to_string())

#Generacion del describe
resumen_sales = sales.describe()
print("Resumen con el metodo describe:\n",resumen_sales)

#Productos clasificados por categoria ???
#TODO

#################################################
### Inventarios

#Rotacion inventarios
rotacion_invent =  (
    sales.groupby(["ID_Tienda","Producto"])["Venta_Total"].sum().div(
        inventories.groupby(["ID_Tienda","Producto"])["Stock_Disponible"].sum()).reset_index(name="Rotacion de inventario"))
print("Rotacion de inventario:\n",rotacion_invent)

#Añadimos la rotacion de inventario al df
inventories = inventories.merge(
    rotacion_invent,
    on=["ID_Tienda", "Producto"],
    how="left"
)
print("Inventario con rotacion\n",inventories)

#Filtrar las tiendas con inventarios críticos (menos del 10% en ventas respecto al inventario disponible).
#unimos los dos df, para asi tener los datos
ventas_inventario = ventas_inventario = sales.merge(
    inventories,
    on=["ID_Tienda", "Producto"],
    how="inner"
)
# Calculamos el porcentaje de ventas respecto al inventario
ventas_inventario["Porcentaje_ventas"] = (
    ventas_inventario["Cantidad_Vendida"]
    / ventas_inventario["Stock_Disponible"]
)

# Filtramos los inferiores al 10%
menor10 = ventas_inventario.query("Porcentaje_ventas < 0.10")
print("Inventarios críticos:\n",menor10)
#Nos dara un dataframe empty , porque no hay ninguno

#################################################
### Satisfacción

# Generamos un resumen de la satisfacción
resumen_satisfaction = satisfaction.describe()

print("Resumen de satisfacción:")
print(resumen_satisfaction)

# Filtramos las tiendas con satisfacción menor al 60%
satisfaccion_baja = satisfaction.query("Satisfacción_Promedio < 60")

print("Tiendas con satisfacción menor al 60%:")
print(satisfaccion_baja)

#################################################
### Calculos

#calculamos la mediana
mediana_ventas = sales["Venta_Total"].median()
print("La mediana de ventas es la siguiente:",mediana_ventas)
#calculamos la desviacion estandar
desviacion = sales["Venta_Total"].std()
print("La desviacion estandar es la siguiente:",desviacion)

#################################################
### Proyección de ventas futuras

# Fijamos una semilla para obtener siempre los mismos valores aleatorios
np.random.seed(42)

# Convertimos las ventas actuales por tienda en un array de NumPy
ventas_actuales = ventas_tienda.values

# Generamos variaciones aleatorias entre -10% y +10% para los próximos 6 meses
variacion = np.random.uniform(
    0.90,
    1.10,
    size=(len(ventas_actuales), 6)
)

# Calculamos las ventas proyectadas
proyeccion_ventas = ventas_actuales[:, np.newaxis] * variacion

# Convertimos el array en un DataFrame
proyeccion_ventas = pd.DataFrame(
    proyeccion_ventas,
    index=ventas_tienda.index,
    columns=[
        "Mes_1",
        "Mes_2",
        "Mes_3",
        "Mes_4",
        "Mes_5",
        "Mes_6"
    ]
)

# Redondeamos a 2 decimales
proyeccion_ventas = proyeccion_ventas.round(2)

print("Proyección de ventas futuras:")
print(proyeccion_ventas)

Cantidad de productos vendidos:
 Producto
Producto A    85
Producto B    75
Producto C    90
Cantidad de productos vendidos , clasificado por tienda:
 ID_Tienda
1    35
2    55
3    50
4    60
5    50
Calculo ingresos por tienda:
 ID_Tienda
1     5000
2    10500
3     9000
4    13000
5    13000
Resumen con el metodo describe:
        ID_Tienda  Cantidad_Vendida  Precio_Unitario   Venta_Total
count  10.000000         10.000000        10.000000     10.000000
mean    3.000000         25.000000       190.000000   5050.000000
std     1.490712          9.128709        87.559504   3361.960407
min     1.000000         10.000000       100.000000   1000.000000
25%     2.000000         20.000000       100.000000   2625.000000
50%     3.000000         25.000000       200.000000   3500.000000
75%     4.000000         30.000000       275.000000   7875.000000
max     5.000000         40.000000       300.000000  10500.000000
Rotacion de inventario:
    ID_Tienda    Producto  Rotacion de inventario
0  